## Ana Paula

In [1]:
import pandas as pd
df = pd.read_csv('data/fake_job_postings.csv')
print(df.shape)
print(df.head())

(17880, 18)
   job_id                                      title            location  \
0       1                           Marketing Intern    US, NY, New York   
1       2  Customer Service - Cloud Video Production      NZ, , Auckland   
2       3    Commissioning Machinery Assistant (CMA)       US, IA, Wever   
3       4          Account Executive - Washington DC  US, DC, Washington   
4       5                        Bill Review Manager  US, FL, Fort Worth   

  department salary_range                                    company_profile  \
0  Marketing          NaN  We're Food52, and we've created a groundbreaki...   
1    Success          NaN  90 Seconds, the worlds Cloud Video Production ...   
2        NaN          NaN  Valor Services provides Workforce Solutions th...   
3      Sales          NaN  Our passion for improving quality of life thro...   
4        NaN          NaN  SpotSource Solutions LLC is a Global Human Cap...   

                                         descripti

In [2]:
# Eliminar columnas que no aportan información al modelo
columnas_a_eliminar = ['job_id'] # El ID es solo un contador
df = df.drop(columns=columnas_a_eliminar)

In [3]:
# Rellenar nulos en columnas categóricas y de texto
categoricas_y_texto = ['department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'employment_type', 'required_experience', 'required_education', 'industry', 'function']

for col in categoricas_y_texto:
    df[col] = df[col].fillna('Unspecified')

In [4]:
import re

def limpiar_texto(texto):
    if pd.isna(texto) or texto == 'Unspecified':
        return ''
    texto = str(texto).lower() # Todo a minúsculas
    texto = re.sub(r'[^a-z0-9\s]', '', texto) # Quitar caracteres especiales (deja solo letras, números y espacios)
    texto = re.sub(r'\s+', ' ', texto).strip() # Limpiar espacios dobles o saltos de línea
    return texto

# Ejemplo: Aplicar la limpieza a la columna 'description'
df['description_clean'] = df['description'].apply(limpiar_texto)

In [5]:
# Extraer el código del país (los dos primeros caracteres antes de la primera coma)
df['country'] = df['location'].apply(lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unspecified')

In [6]:
# Ver la distribución de clases
print(df['fraudulent'].value_counts())
print(df['fraudulent'].value_counts(normalize=True) * 100)

fraudulent
0    17014
1      866
Name: count, dtype: int64
fraudulent
0    95.1566
1     4.8434
Name: proportion, dtype: float64


In [7]:
# 1. Comprobar que ya no quedan nulos en las columnas que vas a usar
print(df[['title', 'description_clean', 'country', 'fraudulent']].isnull().sum())

# 2. Ver cómo ha quedado una fila real tras la limpieza
print(df['description_clean'].iloc[0][:300]) # Muestra los primeros 300 caracteres de la primera descripción

title                0
description_clean    0
country              0
fraudulent           0
dtype: int64
food52 a fastgrowing james beard awardwinning online food community and crowdsourced and curated recipe hub is currently interviewing full and parttime unpaid interns to work in a small team of editors executives and developers in its new york city headquartersreproducing andor repackaging existing 


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# =====================================================================
# 1. CARGA DE DATOS Y FEATURE ENGINEERING (INGENIERÍA DE VARIABLES)
# =====================================================================

# Variable 1: Longitud del texto de la descripción
df['description_length'] = df['description'].astype(str).apply(len)

# Variable 2: ¿Especifica el rango salarial? (1 = Sí, 0 = No)
# Muchos fraudes omiten el salario o usan formatos ocultos
df['has_salary'] = df['salary_range'].notna().astype(int)

# Variable 3: ¿Tiene el perfil de la empresa completo? (1 = Sí, 0 = No)
# Las empresas fantasma suelen dejar este campo vacío
df['has_company_profile'] = df['company_profile'].notna().astype(int)

# Tratamiento de nulos estricto en la descripción para evitar fallos en el NLP
df['description'] = df['description'].fillna('')


# =====================================================================
# 2. PROCESAMIENTO SEMÁNTICO DEL TEXTO (NLP + REDUCCIÓN SVD)
# =====================================================================

print("Procesando texto de las descripciones (TF-IDF)...")
# Convertimos el texto en matriz numérica limitando a las 5,000 palabras más frecuentes
tfidf = TfidfVectorizer(stop_words='english', max_features=5000) 
X_tfidf_completo = tfidf.fit_transform(df['description'])

print("Comprimiendo dimensiones del texto con TruncatedSVD a 50 componentes...")
# Reducimos las miles de columnas de vocabulario a 50 componentes estructurados
svd = TruncatedSVD(n_components=50, random_state=42)
X_texto_reducido = svd.fit_transform(X_tfidf_completo)

# Creamos un DataFrame limpio para los componentes del texto
df_texto_reducido = pd.DataFrame(
    X_texto_reducido, 
    columns=[f'texto_comp_{i}' for i in range(50)]
)


# =====================================================================
# 3. CONSOLIDACIÓN DEL DATASET MIXTO
# =====================================================================

# Lista final de variables clásicas y nuevas variables de negocio
features_clasicas = [
    'telecommuting', 'has_company_logo', 'has_questions', 
    'description_length', 'has_salary', 'has_company_profile'
]

# Reiniciamos índices para asegurar una alineación perfecta al concatenar de lado
X_clasicas = df[features_clasicas].reset_index(drop=True)
df_texto_reducido = df_texto_reducido.reset_index(drop=True)

# Unimos horizontalmente tablas numéricas y componentes de texto
X_final = pd.concat([X_clasicas, df_texto_reducido], axis=1)
y = df['fraudulent'].reset_index(drop=True)

print(f"Dataset consolidado con éxito. Forma final: {X_final.shape}")


# =====================================================================
# 4. DIVISIÓN ROBUSTA (TRAIN / TEST)
# =====================================================================

# Dividimos en 80% entrenamiento y 20% test
# 'stratify=y' es obligatorio para mantener la proporción crítica de fraudes
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)


from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("Entrenando modelo alternativo con XGBoost...")

# 1. Instanciamos XGBoost con parámetros anti-overfitting (árboles cortos y ritmo lento)
modelo_xgb = XGBClassifier(
    n_estimators=100,
    max_depth=4,               # Árboles muy cortos para blindar el overfitting < 5%
    learning_rate=0.08,        # Aprendizaje pausado para generalizar mejor
    scale_pos_weight=19,       # Balancea el desbalanceo (19 reales por cada 1 fraude)
    random_state=42,
    eval_metric='logloss'
)

# 2. Entrenamos con los datos del dataset mixto
modelo_xgb.fit(X_train, y_train)

# =====================================================================
# EVALUACIÓN ESPECÍFICA PARA XGBOOST
# =====================================================================
# Evaluamos primero con el umbral estándar (0.50) para ver cómo se comporta solo
y_pred_xgb = modelo_xgb.predict(X_test)

print("\n" + "="*60)
print(" EVALUACIÓN FINAL DE LA ALTERNATIVA: XGBOOST (UMBRAL 0.50)")
print("="*60)

print("\nMATRIZ DE CONFUSIÓN:")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nREPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test, y_pred_xgb))

Procesando texto de las descripciones (TF-IDF)...
Comprimiendo dimensiones del texto con TruncatedSVD a 50 componentes...
Dataset consolidado con éxito. Forma final: (17880, 56)
Entrenando modelo alternativo con XGBoost...

 EVALUACIÓN FINAL DE LA ALTERNATIVA: XGBOOST (UMBRAL 0.50)

MATRIZ DE CONFUSIÓN:
[[3150  253]
 [  31  142]]

REPORTE DE CLASIFICACIÓN:
              precision    recall  f1-score   support

           0       0.99      0.93      0.96      3403
           1       0.36      0.82      0.50       173

    accuracy                           0.92      3576
   macro avg       0.67      0.87      0.73      3576
weighted avg       0.96      0.92      0.93      3576

